In [ ]:
#BIBLIOTECAS NECESARIAS PARA EJECUTAR CÓDIGOS -------------------------------
import pandas as pd
import numpy as np

In [ ]:
Exportaciones_2025 = pd.read_excel("Base_Unificada.xlsx")

In [ ]:
def clasificar_producto(ncm):
    try:
        limpio = ''.join(filter(str.isdigit, str(ncm)))
        if limpio.startswith(("01", "02", "03", "07", "08", "10", "12", "13", "25", "5201")):  
            return "Primario"
        elif limpio.startswith(("28","29","30","31","32","33","34","35","36", "38","39","40","42","43","44", "45","47","48","49","56", "57","58","59","61","62","63","65","68","69","70","71","72", "73","74","76","80","81", "82","83", "84","85","86","87", "90","92", "94", "95","96","97")):
            return "MOI"
        elif limpio.startswith(("04", "05","09", "11", "15","16", "17", "18","19", "20", "21", "22", "23","41", "5202", "5205")):
            return "MOA"
        elif limpio.startswith("27"):
            return "Energía"
        else:
            return None
    except:
        return None

Exportaciones_2025["Sector"] = Exportaciones_2025["NCM-SIM"].apply(clasificar_producto)

print(Exportaciones_2025[["NCM-SIM", "Sector"]].head(10))


In [ ]:
#FILTRAMOS LOS DATOS POR MES Y LOCALIDAD 
Exportaciones_2025['Fecha'] = pd.to_datetime(Exportaciones_2025['Fecha'])
#Exportaciones_2025_sept_SF = Exportaciones_2025.loc[(Exportaciones_2025['Fecha'].dt.month ==9) & (Exportaciones_2025['Fecha'].dt.year ==2025) &(Exportaciones_2025['Localidad'] == "SANTA FE")] 
print(Exportaciones_2025)

In [ ]:
#TOTAL DE EXPORTACIONES
Exportaciones_SF = Exportaciones_2025.groupby("Localidad")['FOB Divisa'].sum()
print(Exportaciones_SF)

In [ ]:
#EXPORTADORES DE SANTA FE
Exportaciones_por_empresa = Exportaciones_2025.groupby('Exportador')['FOB Divisa'].sum()
Exportaciones_por_empresa = Exportaciones_por_empresa.sort_values(ascending=False)
Exportaciones_por_empresa.to_excel("Lista de empresas por controlar.xlsx")
print(Exportaciones_por_empresa)

In [ ]:
import pandas as pd
from unidecode import unidecode

# Leer los archivos
Empresas_control = pd.read_excel("Lista de empresas por controlar.xlsx")
Empresas_modificadas = pd.read_excel("Lista de empresas modificadas.xlsx")

# --- Normalizar texto para asegurar coincidencias ---
def limpiar_texto(columna):
    return (
        columna.astype(str)
        .apply(unidecode)        
        .str.strip()             
        .str.upper()             
    )

Empresas_control['Exportador'] = limpiar_texto(Empresas_control['Exportador'])
Empresas_modificadas['Exportador'] = limpiar_texto(Empresas_modificadas['Exportador'])

# --- Buscar las empresas de control que no están en modificadas ---
nuevas_empresas = Empresas_control[~Empresas_control['Exportador'].isin(Empresas_modificadas['Exportador'])]

# --- Mostrar resultado ---
print("🚫 Empresas que están en la lista de control pero NO en la lista modificada:")
print(nuevas_empresas['Exportador'])

# (Guardar el resultado en un Excel
nuevas_empresas.to_excel("Empresas_pendientes.xlsx", index=False)
print("\n✅ Archivo guardado como 'Empresas_pendientes.xlsx'")


In [ ]:
#RUBRO DE LOS EXPORTADORES SANTAFESINOS
Exportaciones_por_rubro_SF = Exportaciones_2025.groupby(['Exportador', "Descripcion Arancelaria"], as_index=False)['FOB Divisa'].sum().sort_values(by='FOB Divisa', ascending=False)
total_exportaciones_rubro_SF = Exportaciones_por_rubro_SF["FOB Divisa"].sum()
Exportaciones_por_rubro_SF["Participación (%)"] = (Exportaciones_por_rubro_SF["FOB Divisa"] / total_exportaciones_rubro_SF) * 100
Exportaciones_por_rubro_SF= Exportaciones_por_rubro_SF.sort_values("Participación (%)", ascending=False)
print(Exportaciones_por_rubro_SF)

In [ ]:
#SECTOR DE LOS EXPORTADORES SANTAFESINOS
Exportaciones_por_sector_SF = Exportaciones_2025.groupby(['Sector', "Exportador"], as_index=False)['FOB Divisa'].sum()
total_exportaciones_sector_SF = Exportaciones_por_sector_SF["FOB Divisa"].sum()
Exportaciones_por_sector_SF["Participación (%)"] = (Exportaciones_por_sector_SF["FOB Divisa"] / total_exportaciones_sector_SF) * 100
Exportaciones_por_sector_SF= Exportaciones_por_sector_SF
print(Exportaciones_por_sector_SF)

In [ ]:
#PAISES DE DESTINO DE LOS EXPORTADORES SANTAFESINOS
Exportaciones_por_pais_SF = Exportaciones_2025.groupby(['País de Destino', "Exportador"], as_index=False)['FOB Divisa'].sum()
total_exportaciones_pais_SF = Exportaciones_por_pais_SF["FOB Divisa"].sum()
Exportaciones_por_pais_SF["Participación (%)"] = (Exportaciones_por_pais_SF["FOB Divisa"] / total_exportaciones_pais_SF) * 100
Exportaciones_por_pais_SF= Exportaciones_por_pais_SF
print(Exportaciones_por_pais_SF)

In [ ]:
#DESTINOS DE LAS EXPORTACIONES
Exportaciones_por_pais = Exportaciones_2025.groupby('País de Destino', as_index=False)['FOB Divisa'].sum().sort_values(by='FOB Divisa', ascending=False)
total_exportaciones_pais = Exportaciones_por_pais["FOB Divisa"].sum()
Exportaciones_por_pais["Participación (%)"] = (Exportaciones_por_pais["FOB Divisa"] / total_exportaciones_pais) * 100
Exportaciones_por_pais = Exportaciones_por_pais.sort_values("Participación (%)", ascending=False)
print(Exportaciones_por_pais)

In [ ]:
#DISTRIBUCIÓN DE LAS EXPORTACIONES POR SECTOR
Exportaciones_por_sector = Exportaciones_2025.groupby('Sector', as_index=False)['FOB Divisa'].sum().sort_values(by='FOB Divisa', ascending=False)
total_exportaciones_sector = Exportaciones_por_sector["FOB Divisa"].sum()

Exportaciones_por_sector["Participación (%)"] = (Exportaciones_por_sector["FOB Divisa"] / total_exportaciones_sector) * 100
Exportaciones_por_sector = Exportaciones_por_sector.sort_values("Participación (%)", ascending=False)

print(Exportaciones_por_sector)

In [ ]:
#PRINCIPALES PRODUCTOS EXPORTADOS
Exportaciones_por_rubro = Exportaciones_2025.groupby('Descripcion Arancelaria', as_index=False)['FOB Divisa'].sum().sort_values(by='FOB Divisa', ascending=False)
total_exportaciones_rubro = Exportaciones_por_rubro["FOB Divisa"].sum()

Exportaciones_por_rubro["Participación (%)"] = (Exportaciones_por_rubro["FOB Divisa"] / total_exportaciones_rubro) * 100
Exportaciones_por_rubro = Exportaciones_por_rubro.sort_values("Participación (%)", ascending=False)

print(Exportaciones_por_rubro)

In [ ]:
#EXPORTADORES POR LOCALIDAD
Exportaciones_2025['Fecha'] = pd.to_datetime(Exportaciones_2025['Fecha'])

Exportaciones_por_localidad_e = Exportaciones_2025.groupby(['Localidad', "Exportador"])['FOB Divisa'].sum()
print(Exportaciones_por_localidad_e)

**ANALISIS DE EXPORTACIONES - PRINCIPALMENTE PARA VER EVOLUCIONES TRIMESTRALES Y/O SEMESTRALES**

In [ ]:
#EVOLUCIÓN DE LOS TOTALES DE EXPORTACIONES
Exportaciones_2025_SF = Exportaciones_2025.loc[(Exportaciones_2025['Localidad'] == "SANTA FE")] 

Exportaciones_2025_SF['Fecha'] = pd.to_datetime(Exportaciones_2025_SF['Fecha'])
Exportaciones_2025_SF['Mes'] = Exportaciones_2025_SF['Fecha'].dt.month_name()
Exportaciones_2025_SF['NroMes'] = Exportaciones_2025_SF['Fecha'].dt.month

Exportaciones_2025_totales = (
    Exportaciones_2025_SF
    .groupby(['NroMes', 'Mes'])['FOB Divisa']
    .sum()
    .reset_index()
    .sort_values('NroMes')
)

print(Exportaciones_2025_totales)


In [ ]:
with pd.ExcelWriter('Exportaciones Santa Fe tercer trimestre.xlsx', engine='openpyxl') as writer:
    Exportaciones_SF.to_excel(writer, sheet_name='Total EXPO', index=False)
    Exportaciones_por_empresa.to_excel(writer, sheet_name='Exportadores de SF')
    Exportaciones_por_rubro_SF.to_excel(writer, sheet_name='Rubros de SF', index=False)
    Exportaciones_por_sector_SF.to_excel(writer, sheet_name='Sectores de SF', index=False)
    Exportaciones_por_pais_SF.to_excel(writer, sheet_name='Destinos y empr SF', index=False)
    Exportaciones_por_pais.to_excel(writer, sheet_name='Destino SF', index=False)
    Exportaciones_por_sector.to_excel(writer, sheet_name='Destinos gral', index=False)
    Exportaciones_por_rubro.to_excel(writer, sheet_name='Rubro gral', index=False)
    Exportaciones_por_localidad_e.to_excel(writer, sheet_name='Expo Localidades')
    Exportaciones_2025_totales.to_excel(writer, sheet_name='Evolucion mensual')